In [ ]:
# prompt: montar o google drive, acessar a pasta metaheuristicas e baixar ep.zip no ambiente local, apósisso descompactar e fazer um algoritmo para ler as instancias .dat

# Mount Google Drive
from google.colab import drive
import os # Import os here to use it for path checking
drive.mount('/content/drive')

# Check if Google Drive is mounted
if not os.path.exists('/content/drive'):
    print("Error: Google Drive not mounted. Please check the output of drive.mount().")
else:
    # Change directory to the specified folder in Google Drive
    drive_path = '/content/drive/MyDrive/MetaHeuristicas'
    # Check if the target directory exists before changing into it
    if not os.path.exists(drive_path):
        print(f"Error: The directory '{drive_path}' was not found in your Google Drive.")
        print("Please ensure the path is correct and the directory exists.")
    else:
        os.chdir(drive_path)

        # Download the zip file
        # Check if the zip file exists before trying to copy it
        if not os.path.exists('ep.zip'):
            print("Error: ep.zip not found in the specified Google Drive directory.")
            print(f"Please ensure ep.zip is located at {os.path.join(drive_path, 'ep.zip')}")
        else:
            !cp ep.zip /content/ep.zip

            # Unzip the file
            # Check if the zip file was copied successfully before unzipping
            if not os.path.exists('/content/ep.zip'):
                print("Error: ep.zip was not copied to /content/. Cannot unzip.")
            else:
                !unzip -o /content/ep.zip -d /content/metaheuristicas_unzipped # Use -o to overwrite existing files

                # Now you can write an algorithm to read the .dat files.
                # Assuming the .dat files are now in /content/metaheuristicas_unzipped
                import os

                def read_dat_file(filepath):
                  """Reads a .dat file and returns its content."""
                  with open(filepath, 'r') as f:
                    content = f.read()
                  return content

                # Example of how to iterate through .dat files and read them
                dat_files_directory = '/content/metaheuristicas_unzipped'
                # Check if the unzipped directory exists
                if not os.path.exists(dat_files_directory):
                    print(f"Error: The unzipped directory '{dat_files_directory}' was not created.")
                    print("Please check the unzip command output for errors.")
                else:
                    for filename in os.listdir(dat_files_directory):
                      if filename.endswith(".dat"):
                        filepath = os.path.join(dat_files_directory, filename)
                        file_content = read_dat_file(filepath)
                        print(f"Content of {filename}:\n{file_content}\n---")

                    # You would replace the print statement with your specific logic
                    # to process the data from each .dat file.

Mounted at /content/drive
Archive:  /content/ep.zip
   creating: /content/metaheuristicas_unzipped/instances/
  inflating: /content/metaheuristicas_unzipped/instances/Readme.md  
  inflating: /content/metaheuristicas_unzipped/instances/ep04.dat  
  inflating: /content/metaheuristicas_unzipped/instances/ep01.dat  
  inflating: /content/metaheuristicas_unzipped/instances/ep03.dat  
  inflating: /content/metaheuristicas_unzipped/instances/ep05.dat  
  inflating: /content/metaheuristicas_unzipped/instances/ep02.dat  
  inflating: /content/metaheuristicas_unzipped/instances/ep09.dat  
  inflating: /content/metaheuristicas_unzipped/instances/ep08.dat  
  inflating: /content/metaheuristicas_unzipped/instances/ep06.dat  
  inflating: /content/metaheuristicas_unzipped/instances/ep10.dat  
  inflating: /content/metaheuristicas_unzipped/instances/ep07.dat  


In [ ]:
# prompt: faça uma função que lê e printa os dats

def process_dat_files(directory_path):
  """
  Reads and prints the content of all .dat files in a given directory.

  Args:
    directory_path (str): The path to the directory containing the .dat files.
  """
  # Check if the directory exists
  if not os.path.exists(directory_path):
      print(f"Error: The directory '{directory_path}' was not found.")
      return

  for filename in os.listdir(directory_path):
    if filename.endswith(".dat"):
      filepath = os.path.join(directory_path, filename)
      try:
        with open(filepath, 'r') as f:
          content = f.read()
        print(f"Content of {filename}:\n{content}\n---")
      except Exception as e:
        print(f"Error reading file {filename}: {e}")

# Assuming the .dat files are in /content/metaheuristicas_unzipped after unzipping
process_dat_files('/content/metaheuristicas_unzipped/instances')


In [ ]:
# ingredientes, numero de ingredientes,


#pares


#Peso maior W

"""
Melhores Valores Conhecidos das Instâncias.
Instancia 1 2118
Instancia 2 1378
Instancia 3 2850
Instancia 4 2730
Instancia 5 2624
Instancia 6 4690
Instancia 7 4440
Instancia 8 5020
Instancia 9 4568
Instancia 10 4390
"""

#Cada posição no vetor é uma instância, BKV é seu melhor valor conhecido.
BKV = [2118, 1378, 2850, 2730, 2624, 4690, 4440, 5020, 4568, 4390]




In [ ]:
def read_ep_instance(file_path):
    """
    Lê um arquivo de instância do problema do ensopado perfeito.

    Corrigido para ler sabores/pesos em múltiplas linhas.
    """
    with open(file_path, 'r') as f:
        lines = [line.strip() for line in f.readlines()]

    blocks = []
    current_block = []
    for line in lines:
        if not line:
            if current_block:
                blocks.append(current_block)
                current_block = []
        else:
            current_block.append(line)
    if current_block:
        blocks.append(current_block)

    if len(blocks) < 3:
        raise ValueError("Formato inválido: blocos insuficientes")

    # Bloco 0: n, |I|, W
    n, num_incompatible, W = map(int, blocks[0][0].split())

    # Bloco 1: Sabores (pode ter múltiplas linhas)
    t = []
    for line in blocks[1]:  # Lê todas as linhas do bloco de sabores
        t.extend(map(int, line.split()))
    if len(t) != n:
        raise ValueError(f"Sabores: esperado {n}, encontrado {len(t)}")

    # Bloco 2: Pesos (pode ter múltiplas linhas)
    w = []
    for line in blocks[2]:  # Lê todas as linhas do bloco de pesos
        w.extend(map(int, line.split()))
    if len(w) != n:
        raise ValueError(f"Pesos: esperado {n}, encontrado {len(w)}")

    # Bloco 3: Pares incompatíveis (se existir)
    incompatible_pairs = []
    if len(blocks) >= 4:
        for line in blocks[3]:
            j, k = map(int, line.split())
            incompatible_pairs.append((j-1, k-1))

    if len(incompatible_pairs) != num_incompatible:
        raise ValueError(f"Pares incompatíveis: esperado {num_incompatible}, encontrado {len(incompatible_pairs)}")

    return {
        'n': n,
        'num_incompatible': num_incompatible,
        'W': W,
        't': t,
        'w': w,
        'incompatible_pairs': incompatible_pairs
    }

In [ ]:
import os
import matplotlib.pyplot as plt

local = '/content/metaheuristicas_unzipped/instances'
instancias = [f for f in os.listdir(local) if f.endswith(".dat")]
instancias.sort()

# Lê a primeira instância
inst_path = os.path.join(local, instancias[0])
dados = read_ep_instance(inst_path)

In [ ]:
print(instancias)

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(dados['w'], dados['t'], alpha=0.5)
plt.title(f"Sabor vs. Peso - {instancias[0]}")
plt.xlabel("Peso (w)")
plt.ylabel("Sabor (t)")
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(dados['t'], bins=20, color='skyblue', edgecolor='black')
plt.title(f"Distribuição de Sabores - {instancias[0]}")
plt.xlabel("Sabor")
plt.ylabel("Frequência")
plt.show()

In [ ]:
from collections import defaultdict

# Conta incompatibilidades por ingrediente
incomp_count = defaultdict(int)
for j, k in dados['incompatible_pairs']:
    incomp_count[j] += 1
    incomp_count[k] += 1

# Converte para lista ordenada
counts = [incomp_count[i] for i in range(dados['n'])]

plt.figure(figsize=(10, 6))
plt.bar(range(dados['n']), counts)
plt.title(f"Incompatibilidades por Ingrediente - {instancias[0]}")
plt.xlabel("Ingrediente (ID)")
plt.ylabel("Número de Incompatibilidades")
plt.show()

In [ ]:
df = pd.DataFrame({
    'weight': dados['w'],
    'taste': dados['t']
})

# Now 'df' is a DataFrame where each row is an ingredient
# and the columns are 'weight' and 'taste'.
print(df.head(500)) # Print the head of the DataFrame to verify